In [47]:
import shap
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [48]:
# --- 3. Greedy Feature Selection (For Comparison) ---
# Define the characteristic function v(S) for the predicted class
def v_x(S, x=x_q, background=X_train, model=model, pred_class=query_pred_class):
    """Compute E[f(X)|X_S=x_S] by replacing other features with background samples."""
    X_masked = background.copy()
    for i in S:
        X_masked[feature_names[i]] = x.iloc[0, i]
    preds = model.predict_proba(X_masked)[:, pred_class]
    return preds.mean()

In [49]:
# --- 0. Configuration ---
QUERY_INSTANCE_IDX = 3 # We'll pick a query instance from X_test (index 3)
TOP_K_LOCAL = 1      # Number of features to pick based on local SHAP importance
TOP_K_COVERAGE = 5   # Top features per instance to define 'coverage' in the Set Cover problem
NUM_COVER_FEATURES = 4 # Number of additional features to pick based on Max Coverage

In [50]:
# --- 1. Setup and Training ---
X, y = load_breast_cancer(return_X_y=True, as_frame=True)
feature_names = X.columns
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [51]:
# --- 2. Query Instance & Local Analysis (from min_feat.py) ---
x_q = X_test.iloc[[QUERY_INSTANCE_IDX]]
query_pred_class = model.predict(x_q)[0]
query_target_value = model.predict_proba(x_q)[0, query_pred_class]
contrast_class_label = 1 - query_pred_class
class_names = {0: "Malignant (0)", 1: "Benign (1)"}

print("=" * 60)
print(f"QUERY INSTANCE ANALYSIS (Index {QUERY_INSTANCE_IDX}):")
print(f"Predicted Class: {class_names[query_pred_class]} (P={query_target_value:.4f})")
print(f"Contrast Class for Coverage: {class_names[contrast_class_label]}")
print("=" * 60)

QUERY INSTANCE ANALYSIS (Index 3):
Predicted Class: Benign (1) (P=0.9900)
Contrast Class for Coverage: Malignant (0)


In [52]:
# Compute SHAP values for the query instance
explainer = shap.TreeExplainer(model, X_train)
shap_values_q = explainer.shap_values(x_q)
# Select the SHAP values corresponding to the predicted class
phi_q = shap_values_q[0][:, query_pred_class]

# --- 2.1 Local SHAP Ranking ---
abs_phi_q = np.abs(phi_q)
shap_ranking_q_idx = np.argsort(abs_phi_q)[::-1]  # indices of features, sorted by descending absolute SHAP value

# Select the most important local feature
local_best_feature_idx = shap_ranking_q_idx[:TOP_K_LOCAL]
print("local_best_feature_idx:", local_best_feature_idx)
local_best_feature_name = feature_names[local_best_feature_idx].tolist()

print("\n--- A. Local SHAP Importance ---")
print(f"Top {TOP_K_LOCAL} Feature(s) for Query Prediction:")
for i in local_best_feature_idx:
    print(f"- {feature_names[i]} (SHAP: {phi_q[i]:.4f})")


local_best_feature_idx: [23]

--- A. Local SHAP Importance ---
Top 1 Feature(s) for Query Prediction:
- worst area (SHAP: 0.0663)


In [53]:
# Run Greedy Selection (using logic from min_feat.py)
baseline_value = v_x([])
current_S = set()
current_value = baseline_value
selected_greedy = []

for _ in range(len(feature_names)):
    best_gain, best_i = 0, None
    for i in range(len(feature_names)):
        if i in current_S:
            continue
        candidate_value = v_x(current_S | {i})
        gain = candidate_value - current_value
        if gain > best_gain:
            best_gain, best_i = gain, i

    if best_i is None:
        break
    current_S.add(best_i)
    current_value = v_x(current_S)
    selected_greedy.append(best_i)
    if abs(current_value - query_target_value) < 0.01:
        break

print("\n--- B. Local Greedy Selection (Comparison) ---")
print(f"Features required to approximate prediction (within 0.01): {len(selected_greedy)}")
print(f"Selected Features: {[feature_names[i] for i in selected_greedy]}")



--- B. Local Greedy Selection (Comparison) ---
Features required to approximate prediction (within 0.01): 17
Selected Features: ['mean concave points', 'worst area', 'worst radius', 'worst concave points', 'worst perimeter', 'mean radius', 'mean perimeter', 'area error', 'mean area', 'worst texture', 'radius error', 'worst concavity', 'mean texture', 'worst compactness', 'mean concavity', 'compactness error', 'perimeter error']


In [54]:

# Filter training data for the contrast class
X_contrast = X_train[y_train == contrast_class_label]
if X_contrast.empty:
    print("\nError: No samples found for the contrast class in the training set.")
    exit()
n_features = X_contrast.shape[1]    
# Compute SHAP values for the entire contrast set
shap_values_contrast_all = explainer.shap_values(X_contrast)

# Select SHAP values corresponding to the CONTRAST class
if shap_values_contrast_all.shape[2] == len(np.unique(y_train)):  # classes on last axis
    shap_values_contrast = shap_values_contrast_all[:, :, contrast_class_label]   # -> (n_samples, n_features)
elif shap_values_contrast_all.shape[0] == len(np.unique(y_train)):  # classes on first axis
    shap_values_contrast = shap_values_contrast_all[contrast_class_label, :, :]   # -> (n_samples, n_features)
else:
    n_samples = X_contrast.shape[0]
    if shap_values_contrast_all.shape[0] == n_samples and shap_values_contrast_all.shape[1] != n_features and shap_values_contrast_all.shape[2] == n_features:
        shap_values_contrast = shap_values_contrast_all[:, :, contrast_class_label]
print("shap_values_contrast shape:", shap_values_contrast.shape)

n_instances, n_features = shap_values_contrast.shape
coverage_sets = [set() for _ in range(n_features)]  # one set per feature [set() for _ in range(n_features)]  # one set per feature

print(shap_values_contrast[0])
# 4.1 Define Coverage
for j in range(n_instances):
    # Find the TOP_K_COVERAGE features by absolute SHAP value for instance j
    top_features_idx = np.argsort(-np.abs(shap_values_contrast[j]))[:TOP_K_COVERAGE]
    if j == 0:  # Print for first 5 instances for verification
        print("top_features_idx for instance", j, ":", top_features_idx)
    for i in top_features_idx:
        coverage_sets[i].add(j)  # Feature i covers contrast instance j

shap_values_contrast shape: (169, 30)
[ 0.0182948   0.01365111  0.0149269   0.02536766  0.00662     0.00219825
  0.03464595  0.05801492  0.00361762 -0.00124734  0.01740329  0.00079893
  0.00728071  0.0287919   0.00210417 -0.00471937  0.0053973  -0.00063341
 -0.00031048  0.00209381  0.04335171  0.02406671  0.06182575  0.08579658
  0.0089506   0.01738492  0.04325468  0.08238361  0.01267536  0.00601333]
top_features_idx for instance 0 : [23 27 22  7 20]


In [55]:
# --- 4. Max Coverage on Contrast Class (from max_cover.py) ---
#for each feature in all features compute how many contrast instances it covers

coverage_counts = [len(s) for s in coverage_sets]
coverage_percent = [c / n_instances for c in coverage_counts]

# Build a small DataFrame for easy inspection and sorting
coverage_df = pd.DataFrame({
    "idx": np.arange(n_features),
    "feature": feature_names,
    "count": coverage_counts,
    "percent": coverage_percent
})

coverage_df_sorted = coverage_df.sort_values("count", ascending=False).reset_index(drop=True)

print("Top features by contrast-class coverage (count, percent):")
print(coverage_df_sorted.head(10).to_string(index=False))

# If you want the top-K features that maximize coverage  
coverage_topk_indices = coverage_df_sorted.iloc[:NUM_COVER_FEATURES]["idx"].tolist()
print(coverage_topk_indices)
coverage_topk_names = [feature_names[i] for i in coverage_topk_indices]

print(f"\nTop {NUM_COVER_FEATURES} feature indices (max coverage): {coverage_topk_indices}")
print(f"Top {NUM_COVER_FEATURES} feature names (max coverage): {coverage_topk_names}")

# get TOP_K_LOCAL local best feature indices + NUM_COVER_FEATURES from coverage_topk_indices
    
hybrid_features_indices = list(local_best_feature_idx)  # start with local best feature(s)
hybrid_features_indices.extend(coverage_topk_indices)
hybrid_features_indices = list(set(hybrid_features_indices))  # ensure uniqueness


Top features by contrast-class coverage (count, percent):
 idx              feature  count  percent
   7  mean concave points    144 0.852071
  23           worst area    144 0.852071
  27 worst concave points    138 0.816568
  22      worst perimeter    137 0.810651
  20         worst radius    136 0.804734
  26      worst concavity     33 0.195266
  21        worst texture     25 0.147929
   6       mean concavity     19 0.112426
   3            mean area     14 0.082840
  13           area error     14 0.082840
[7, 23, 27, 22]

Top 4 feature indices (max coverage): [7, 23, 27, 22]
Top 4 feature names (max coverage): ['mean concave points', 'worst area', 'worst concave points', 'worst perimeter']


In [56]:
# --- 5. Final Hybrid Results ---
print("\n" + "=" * 60)
print("--- C. HYBRID FEATURE SELECTION RESULTS ---")
print(f"Query Predicted Class: {class_names[query_pred_class]}")
print(f"Contrast Class SHAP Coverage Target: {class_names[contrast_class_label]}")
print(f"Number of Features Selected: {len(hybrid_features_indices)}")
print("-" * 60)

print("1. Initial Feature (Local Importance):")
print(f"-> {local_best_feature_name[0]}")

print("\n2. Additional Features (Max Coverage on Contrast Class):")
for i in coverage_topk_indices:
    print(f"-> {feature_names[i]} (coverage gain: {coverage_counts[i]})")



--- C. HYBRID FEATURE SELECTION RESULTS ---
Query Predicted Class: Benign (1)
Contrast Class SHAP Coverage Target: Malignant (0)
Number of Features Selected: 4
------------------------------------------------------------
1. Initial Feature (Local Importance):
-> worst area

2. Additional Features (Max Coverage on Contrast Class):
-> mean concave points (coverage gain: 144)
-> worst area (coverage gain: 144)
-> worst concave points (coverage gain: 138)
-> worst perimeter (coverage gain: 137)


In [57]:
#print the top features by shap values
for i in shap_ranking_q_idx[:15]: print(feature_names[i])

worst area
worst concave points
worst perimeter
worst radius
mean concave points
area error
worst texture
radius error
mean area
worst concavity
mean perimeter
mean radius
mean concavity
worst symmetry
compactness error


In [58]:
# ...existing code...
# New comparison cell: compare top local SHAP features (cell 11) vs final hybrid set
top_local = [feature_names[i] for i in shap_ranking_q_idx[:15]]
hybrid = [feature_names[i] for i in hybrid_features_indices]  # created in the "Final Hybrid Results" cell

intersection = [f for f in top_local if f in hybrid]
only_local = [f for f in top_local if f not in hybrid]
only_hybrid = [f for f in hybrid if f not in top_local]

print("Top local (cell 11) — top 15 SHAP features:")
print(top_local)
print("\nFinal hybrid feature set:")
print(hybrid)
print("\nIntersection (features in both):")
print(intersection)
print("\nOnly in local (not in hybrid):")
print(only_local)
print("\nOnly in hybrid (not in top-15 local):")
print(only_hybrid)
# ...existing code...

# modify the shap explanation to only keep the hybrid features
for i in range(len(phi_q)):
    if i not in hybrid_features_indices:
        phi_q[i] = 0.0
print("\nModified SHAP values for query instance (only hybrid features kept):")
print(phi_q)

Top local (cell 11) — top 15 SHAP features:
['worst area', 'worst concave points', 'worst perimeter', 'worst radius', 'mean concave points', 'area error', 'worst texture', 'radius error', 'mean area', 'worst concavity', 'mean perimeter', 'mean radius', 'mean concavity', 'worst symmetry', 'compactness error']

Final hybrid feature set:
['worst concave points', 'mean concave points', 'worst perimeter', 'worst area']

Intersection (features in both):
['worst area', 'worst concave points', 'worst perimeter', 'mean concave points']

Only in local (not in hybrid):
['worst radius', 'area error', 'worst texture', 'radius error', 'mean area', 'worst concavity', 'mean perimeter', 'mean radius', 'mean concavity', 'worst symmetry', 'compactness error']

Only in hybrid (not in top-15 local):
[]

Modified SHAP values for query instance (only hybrid features kept):
[0.         0.         0.         0.         0.         0.
 0.         0.03312202 0.         0.         0.         0.
 0.         0.     